# NBA Next-Game Points Predictor

Standalone predictor notebook. Loads cached data, retrains the V2 XGBoost model,  
fetches live 2025-26 season data, and forecasts a player's points in their next game.

**To make a prediction:** run all cells top-to-bottom, then edit the variables in the  
last cell (`PLAYER_ID`, `OPPONENT`, `HOME_GAME`, `DAYS_REST`, `TARGET_PTS`) and re-run it.

**Dependencies (must exist before running):**
- `../data/processed/multi_player_logs.json` — game logs for 31 players
- `../data/processed/team_ratings.json` — season-level team ratings
- `../data/processed/team_ratings_rolling.json` — monthly snapshot ratings

In [ ]:
import json, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from scipy.stats import gaussian_kde
from sklearn.metrics import mean_squared_error
from nba_api.stats.static import teams as nba_teams
from nba_api.stats.endpoints import playergamelogs

# ── 31-player roster ──────────────────────────────────────────────────────────
PLAYERS = {
    2544:    'LeBron James',      201939:  'Stephen Curry',
    201142:  'Kevin Durant',      201935:  'James Harden',
    202695:  'Kawhi Leonard',     203507:  'Giannis Antetokounmpo',
    203999:  'Nikola Jokic',      203954:  'Joel Embiid',
    1628369: 'Jayson Tatum',      202710:  'Jimmy Butler',
    202331:  'Paul George',       203081:  'Damian Lillard',
    101108:  'Chris Paul',        201566:  'Russell Westbrook',
    202681:  'Kyrie Irving',      1626164: 'Devin Booker',
    203076:  'Anthony Davis',     1626157: 'Karl-Anthony Towns',
    203497:  'Rudy Gobert',       1628378: 'Donovan Mitchell',
    1629027: 'Trae Young',        1629630: 'Ja Morant',
    1629627: 'Zion Williamson',   1628983: 'Shai Gilgeous-Alexander',
    1630162: 'Anthony Edwards',   1641705: 'Victor Wembanyama',
    1627759: 'Jaylen Brown',      203078:  'Bradley Beal',
    201942:  'DeMar DeRozan',     1627783: 'Pascal Siakam',
    1629029: 'Luka Doncic',
}

SEASONS = {
    '2018-19': '2018_19', '2019-20': '2019_20', '2020-21': '2020_21',
    '2021-22': '2021_22', '2022-23': '2022_23', '2023-24': '2023_24',
    '2024-25': '2024_25',
}

FEATURES_PRED = [
    'season_avg_so_far', 'form_vs_season',
    'roll5_pts', 'roll10_pts',
    'roll5_ast', 'roll5_tov', 'roll5_min',
    'HOME', 'b2b', 'days_rest',
    'opp_def_rating', 'opp_off_rating',
    'games_on_new_team',
]

PARAMS = dict(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
    reg_alpha=0.5, reg_lambda=2.0, objective='reg:squarederror',
    random_state=42, n_jobs=-1,
)

_static        = nba_teams.get_teams()
NBA_ABBR_TO_ID = {t['abbreviation']: t['id'] for t in _static}
ID_TO_ABBR     = {t['id']: t['abbreviation'] for t in _static}

print(f"Imports done. {len(PLAYERS)} players | {len(SEASONS)} seasons.")


In [ ]:
# ── Load cached game logs ─────────────────────────────────────────────────────
with open('../data/processed/multi_player_logs.json') as f:
    raw = json.load(f)

player_dfs = {}
for pid_str, season_dict in raw.items():
    pid = int(pid_str)
    player_dfs[pid] = {}
    for season_key, records in season_dict.items():
        if records:
            df = pd.DataFrame(records)
            df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
            player_dfs[pid][season_key] = df

# ── Load season-level team ratings ────────────────────────────────────────────
with open('../data/processed/team_ratings.json') as f:
    team_ratings = {s: {int(k): v for k, v in d.items()} for s, d in json.load(f).items()}

# ── Load monthly-snapshot rolling ratings ─────────────────────────────────────
with open('../data/processed/team_ratings_rolling.json') as f:
    rolling_ratings = {
        season: {snap: {int(k): v for k, v in d.items()} for snap, d in snaps.items()}
        for season, snaps in json.load(f).items()
    }

total_games = sum(len(df) for pd_ in player_dfs.values() for df in pd_.values())
print(f"Loaded {len(player_dfs)} players — {total_games:,} games")
print(f"Team ratings: {len(team_ratings)} seasons")
print(f"Rolling ratings: {len(rolling_ratings)} seasons")


In [ ]:
import math

def extract_opp_abbr(matchup_str):
    parts = re.split(r'\s+(?:vs\.|@)\s+', str(matchup_str))
    return parts[1].strip() if len(parts) == 2 else None

def parse_min(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return np.nan
    s = str(val).strip()
    if ':' in s:
        parts = s.split(':')
        try:
            return float(parts[0]) + float(parts[1]) / 60
        except ValueError:
            return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

def get_rolling_opp_rating(game_date, season_key, opp_abbr, key):
    tid = NBA_ABBR_TO_ID.get(opp_abbr)
    if not tid:
        return np.nan
    season_snaps = rolling_ratings.get(season_key, {})
    eligible = [d for d in season_snaps if pd.Timestamp(d) <= game_date]
    if eligible:
        return season_snaps[max(eligible)].get(tid, {}).get(key, np.nan)
    return team_ratings.get(season_key, {}).get(tid, {}).get(key, np.nan)

def build_features_v2(pid, season_key, df_in):
    df = df_in.copy().sort_values('GAME_DATE').reset_index(drop=True)
    df['team_changed'] = (df['TEAM_ABBREVIATION'] != df['TEAM_ABBREVIATION'].shift(1)).astype(int)
    df.loc[0, 'team_changed'] = 0
    df['trade_block']       = df['team_changed'].cumsum()
    df['games_on_new_team'] = df.groupby('trade_block').cumcount()
    df['HOME']      = (~df['MATCHUP'].str.contains('@')).astype(int)
    df['days_rest'] = df['GAME_DATE'].diff().dt.days.clip(upper=14).fillna(2)
    df['b2b']       = (df['days_rest'] == 1).astype(int)
    df['MIN_num']   = df['MIN'].apply(parse_min)
    for col in ['PTS', 'AST', 'TOV']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['roll5_pts']  = df['PTS'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll10_pts'] = df['PTS'].shift(1).rolling(10, min_periods=5).mean()
    df['roll5_ast']  = df['AST'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll5_tov']  = df['TOV'].shift(1).rolling(5,  min_periods=3).mean()
    df['roll5_min']  = df['MIN_num'].shift(1).rolling(5, min_periods=3).mean()
    df['season_avg_so_far'] = df['PTS'].shift(1).expanding().mean()
    df['form_vs_season']    = df['roll5_pts'] - df['season_avg_so_far']
    def opp_roll(row, key):
        abbr = extract_opp_abbr(row['MATCHUP'])
        return get_rolling_opp_rating(row['GAME_DATE'], season_key, abbr, key) if abbr else np.nan
    df['opp_def_rating'] = df.apply(lambda r: opp_roll(r, 'DEF'), axis=1)
    df['opp_off_rating'] = df.apply(lambda r: opp_roll(r, 'OFF'), axis=1)
    df['player_id']   = pid
    df['player_name'] = PLAYERS.get(pid, str(pid))
    df['season_key']  = season_key
    return df

print("Feature engineering helpers defined.")


In [ ]:
# ── Build training dataset and train model_pred ───────────────────────────────
# This takes ~60 seconds on first run (feature engineering across 15k games).

all_dfs = []
for pid, season_dict in player_dfs.items():
    for season_key, df_season in season_dict.items():
        try:
            all_dfs.append(build_features_v2(pid, season_key, df_season))
        except Exception as e:
            print(f"  ERROR {PLAYERS.get(pid)} {season_key}: {e}")

df_all   = pd.concat(all_dfs, ignore_index=True).sort_values('GAME_DATE').reset_index(drop=True)
df_train = df_all.dropna(subset=FEATURES_PRED + ['PTS']).copy().reset_index(drop=True)

SPLIT_DATE = pd.Timestamp('2023-10-01')
tr_mask = df_train['GAME_DATE'] < SPLIT_DATE
te_mask = ~tr_mask

X_tr = df_train.loc[tr_mask, FEATURES_PRED].values
y_tr = df_train.loc[tr_mask, 'PTS'].values
X_te = df_train.loc[te_mask, FEATURES_PRED].values
y_te = df_train.loc[te_mask, 'PTS'].values

model_pred = xgb.XGBRegressor(**PARAMS)
model_pred.fit(X_tr, y_tr)

y_hat_te         = model_pred.predict(X_te)
global_residuals = y_te - y_hat_te

print(f"model_pred trained — Test RMSE: {np.sqrt(mean_squared_error(y_te, y_hat_te)):.2f} pts")
print(f"Global residual std: {global_residuals.std():.2f} pts")

# Per-player residual distributions (min 20 test games)
df_te_ref = df_train[te_mask].copy().reset_index(drop=True)
df_te_ref['resid'] = global_residuals
player_residuals = {
    pid: grp['resid'].values
    for pid, grp in df_te_ref.groupby('player_id')
    if len(grp) >= 20
}
print(f"Per-player residuals available for {len(player_residuals)}/{df_te_ref['player_id'].nunique()} players")


In [ ]:
# ── Fetch 2025-26 season team ratings ────────────────────────────────────────
#
# We need current-season opponent ratings for the prediction context.
# These are fetched once, added to rolling_ratings and team_ratings,
# and cached in memory for the session (no file write needed — just re-run
# this cell at the start of each session to get fresh ratings).

from nba_api.stats.endpoints import leaguedashteamstats as _ldt

CURRENT_SEASON     = '2025-26'
CURRENT_SEASON_KEY = '2025_26'

df_curr_ratings = _ldt.LeagueDashTeamStats(
    season=CURRENT_SEASON,
    measure_type_detailed_defense='Advanced',
).get_data_frames()[0]

_snapshot_key = pd.Timestamp.today().strftime('%Y-%m-%d')

# Add to rolling_ratings so build_features_v2 can look up opponent ratings
if CURRENT_SEASON_KEY not in rolling_ratings:
    rolling_ratings[CURRENT_SEASON_KEY] = {}

rolling_ratings[CURRENT_SEASON_KEY][_snapshot_key] = {
    int(row['TEAM_ID']): {'DEF': row['DEF_RATING'], 'OFF': row['OFF_RATING']}
    for _, row in df_curr_ratings.iterrows()
}

# Also add to team_ratings as the season-level fallback
team_ratings[CURRENT_SEASON_KEY] = {
    int(row['TEAM_ID']): {'DEF': row['DEF_RATING'], 'OFF': row['OFF_RATING']}
    for _, row in df_curr_ratings.iterrows()
}

# Build abbr -> ratings lookup for the prediction function
_id_to_abbr_map = {t['id']: t['abbreviation'] for t in nba_teams.get_teams()}
CURRENT_TEAM_RATINGS = {}   # {abbr: {DEF, OFF}}  — used in predict_next_game
for tid, ratings in team_ratings[CURRENT_SEASON_KEY].items():
    abbr = _id_to_abbr_map.get(tid)
    if abbr:
        CURRENT_TEAM_RATINGS[abbr] = ratings

print(f"2025-26 team ratings loaded ({len(CURRENT_TEAM_RATINGS)} teams, snapshot: {_snapshot_key})")
print(f"\nTop-5 toughest defences (highest DEF rating = easier to score against):")
def_rank = sorted(CURRENT_TEAM_RATINGS.items(), key=lambda x: x[1]['DEF'])
print("  Hardest to score against:")
for abbr, r in def_rank[:5]:
    print(f"    {abbr:<5}  DEF={r['DEF']:.1f}")
print("  Easiest to score against:")
for abbr, r in def_rank[-5:]:
    print(f"    {abbr:<5}  DEF={r['DEF']:.1f}")

time.sleep(0.3)

# ── Function: fetch a player's 2025-26 game logs and build v2 features ────────
def fetch_player_current_season(player_id):
    """
    Fetches 2025-26 game logs for player_id and returns a feature-engineered
    DataFrame using build_features_v2.  Raises ValueError if no data found.
    """
    logs = playergamelogs.PlayerGameLogs(
        player_id_nullable=player_id,
        season_nullable=CURRENT_SEASON,
    ).get_data_frames()[0]

    if logs.empty:
        raise ValueError(f"No {CURRENT_SEASON} data found for player ID {player_id}")

    keep = [c for c in ['GAME_DATE', 'TEAM_ABBREVIATION', 'MATCHUP', 'WL',
                         'MIN', 'PTS', 'REB', 'AST', 'TOV', 'PLUS_MINUS']
            if c in logs.columns]
    df = logs[keep].copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    return build_features_v2(player_id, CURRENT_SEASON_KEY, df)

print("\nfetch_player_current_season() defined.")


In [ ]:
# ── predict_next_game — main prediction function ─────────────────────────────
#
# Parameters
# ----------
# player_id      : int   NBA player ID  (see PLAYERS dict for valid IDs)
# opponent_abbr  : str   Opponent team abbreviation e.g. 'BOS', 'LAL', 'GSW'
#                        Pass None to use league-average opponent ratings
# is_home        : bool  True = home game, False = away game
# days_rest_next : int   Days since last game (1 = back-to-back, 2 = normal, 3+ = rested)
# target_pts     : float Compute P(score >= target_pts)  — set to None to skip
# plot           : bool  Show probability density plot of predicted scoring range
#
# How the probability is computed
# --------------------------------
# 1. Get the point prediction μ from model_pred
# 2. Shift the empirical residual distribution by μ → distribution of possible outcomes
# 3. P(score ≥ X) = fraction of (μ + residuals) that exceed X
# 4. The 80% interval is the [p10, p90] of the same shifted distribution

from scipy.stats import gaussian_kde

def predict_next_game(player_id, opponent_abbr=None, is_home=True,
                      days_rest_next=2, target_pts=None, plot=True):

    player_name = PLAYERS.get(player_id, f"Player {player_id}")

    # 1. Fetch and build features from live 2025-26 data
    print(f"Fetching {CURRENT_SEASON} logs for {player_name}...")
    df_live = fetch_player_current_season(player_id)
    df_live = df_live.dropna(subset=FEATURES_PRED)

    if len(df_live) < 5:
        print(f"  Only {len(df_live)} usable rows — not enough to make a reliable prediction.")
        return None

    # 2. Use most recent game's rolling features as the base
    #    (these encode the player's CURRENT form — roll5_pts, season_avg, etc.)
    last_row   = df_live.iloc[-1]
    feat_dict  = {f: float(last_row[f]) for f in FEATURES_PRED}

    # 3. Override forward-looking context with next-game inputs
    feat_dict['HOME']      = 1.0 if is_home else 0.0
    feat_dict['days_rest'] = float(days_rest_next)
    feat_dict['b2b']       = 1.0 if days_rest_next == 1 else 0.0

    if opponent_abbr and opponent_abbr.upper() in CURRENT_TEAM_RATINGS:
        opp = CURRENT_TEAM_RATINGS[opponent_abbr.upper()]
        feat_dict['opp_def_rating'] = opp['DEF']
        feat_dict['opp_off_rating'] = opp['OFF']
        opp_label = opponent_abbr.upper()
    else:
        # League-average ratings
        feat_dict['opp_def_rating'] = float(np.mean([v['DEF'] for v in CURRENT_TEAM_RATINGS.values()]))
        feat_dict['opp_off_rating'] = float(np.mean([v['OFF'] for v in CURRENT_TEAM_RATINGS.values()]))
        opp_label = 'league average'

    # 4. Point prediction
    X_new      = np.array([[feat_dict[f] for f in FEATURES_PRED]])
    point_pred = float(max(0, model_pred.predict(X_new)[0]))

    # 5. Outcome distribution = point_pred + residuals
    resids    = player_residuals.get(player_id, global_residuals)
    outcomes  = point_pred + resids
    outcomes  = outcomes[outcomes >= 0]

    lo80 = float(np.percentile(outcomes, 10))
    hi80 = float(np.percentile(outcomes, 90))

    # 6. Probability that actual score exceeds target
    prob_val  = None
    prob_line = ""
    if target_pts is not None:
        prob_val  = float((outcomes >= target_pts).mean())
        prob_line = f"  P(score ≥ {int(target_pts)} pts)   : {prob_val*100:.1f}%"

    # 7. Print summary
    games_played = len(df_live)
    recent = df_live.tail(5)[['GAME_DATE', 'MATCHUP', 'PTS']].copy()
    recent['GAME_DATE'] = recent['GAME_DATE'].dt.strftime('%Y-%m-%d')

    print(f"\n{'='*58}")
    print(f"  Next Game Prediction — {player_name}")
    print(f"{'='*58}")
    print(f"  Opponent              : {opp_label}")
    print(f"  Location              : {'Home' if is_home else 'Away'}")
    print(f"  Days rest             : {days_rest_next}")
    print(f"  Games played (season) : {games_played}")
    print(f"{'─'*58}")
    print(f"  Predicted pts         : {point_pred:.1f}")
    print(f"  80% interval          : {lo80:.1f} – {hi80:.1f} pts")
    print(f"{'─'*58}")
    print(f"  Season avg so far     : {last_row['season_avg_so_far']:.1f} pts")
    print(f"  Roll-5 form           : {last_row['roll5_pts']:.1f} pts")
    print(f"  Roll-10 form          : {last_row['roll10_pts']:.1f} pts")
    print(f"  Roll-5 minutes        : {last_row['roll5_min']:.1f} min")
    if prob_line:
        print(f"{'─'*58}")
        print(prob_line)
    print(f"{'─'*58}")
    print(f"  Last 5 games:")
    for _, r in recent.iterrows():
        print(f"    {r['GAME_DATE']}  {str(r['MATCHUP']):<30}  {int(r['PTS'])} pts")
    print(f"{'='*58}")

    # 8. Optional probability density plot
    if plot:
        fig, ax = plt.subplots(figsize=(10, 4))
        kde = gaussian_kde(outcomes, bw_method=0.4)
        x_range = np.linspace(max(0, outcomes.min() - 5), outcomes.max() + 5, 300)
        y_vals  = kde(x_range)

        ax.plot(x_range, y_vals, color='steelblue', lw=2, label='Predicted distribution')
        ax.fill_between(x_range, y_vals,
                        where=(x_range >= lo80) & (x_range <= hi80),
                        alpha=0.25, color='steelblue', label=f'80% interval ({lo80:.0f}–{hi80:.0f} pts)')

        ax.axvline(point_pred, color='navy', linestyle='--', lw=2,
                   label=f'Prediction: {point_pred:.1f} pts')

        if target_pts is not None:
            ax.axvline(target_pts, color='crimson', linestyle=':', lw=2,
                       label=f'Target ≥{int(target_pts)} pts  ({prob_val*100:.1f}%)')
            ax.fill_between(x_range, y_vals,
                            where=(x_range >= target_pts),
                            alpha=0.2, color='crimson')

        ax.set_xlabel('Points scored', fontsize=12)
        ax.set_ylabel('Probability density', fontsize=12)
        ax.set_title(f'{player_name} — Next Game Prediction vs {opp_label}', fontsize=13)
        ax.set_xlim(left=0)
        ax.legend(fontsize=10)
        plt.tight_layout()
        plt.show()

    return {
        'player':        player_name,
        'predicted_pts': round(point_pred, 1),
        'interval':      (round(lo80, 1), round(hi80, 1)),
        'prob_exceed':   round(prob_val * 100, 1) if prob_val is not None else None,
    }

print("predict_next_game() ready.")
print(f"\nRegistered players:")
for pid, name in sorted(PLAYERS.items(), key=lambda x: x[1]):
    print(f"  {pid:>8}  {name}")


In [ ]:
# ── Player ID lookup — type any part of the name in lowercase ─────────────────

def find_player(query):
    q = query.strip().lower()
    matches = {pid: name for pid, name in PLAYERS.items() if q in name.lower()}
    if not matches:
        print(f"No match for '{query}'")
    else:
        for pid, name in sorted(matches.items(), key=lambda x: x[1]):
            print(f"  {pid:>8}  {name}")

find_player("luka")


In [ ]:
# ── Run the predictor ─────────────────────────────────────────────────────────
#
# Set PLAYER_ID and TARGET_PTS, then run.
# Opponent, home/away, and days rest are fetched automatically from the NBA schedule.
# Override them manually only if you want to model a hypothetical match-up.

from nba_api.stats.endpoints import scoreboardv2
from datetime import date, timedelta

def fetch_next_scheduled_game(player_id):
    """
    Look up the player's next scheduled game from the live NBA schedule.

    Strategy:
      1. Pull the player's 2025-26 logs to get their current team and last game date.
      2. Walk forward from today, calling ScoreboardV2 for each date (up to 14 days).
      3. Return the first date where that team appears on the schedule.

    Returns a dict: {date, opponent_abbr, is_home, days_rest} or None if not found.
    """
    logs = playergamelogs.PlayerGameLogs(
        player_id_nullable=player_id,
        season_nullable=CURRENT_SEASON,
    ).get_data_frames()[0]

    if logs.empty:
        raise ValueError(f"No {CURRENT_SEASON} data for player {player_id}")

    logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE'])
    logs = logs.sort_values('GAME_DATE')
    last_game  = logs.iloc[-1]
    team_abbr  = last_game['TEAM_ABBREVIATION']
    last_date  = last_game['GAME_DATE']

    print(f"  Current team: {team_abbr}  |  Last game: {last_date.strftime('%Y-%m-%d')}")
    print(f"  Scanning schedule from {date.today()} ...")

    for offset in range(15):
        check_date = date.today() + timedelta(days=offset)
        date_str   = check_date.strftime('%m/%d/%Y')   # ScoreboardV2 format

        try:
            sb     = scoreboardv2.ScoreboardV2(game_date=date_str, league_id='00')
            header = sb.get_data_frames()[0]            # GameHeader table
        except Exception:
            time.sleep(0.4)
            continue

        if header.empty:
            time.sleep(0.3)
            continue

        for _, game in header.iterrows():
            home_abbr = ID_TO_ABBR.get(int(game['HOME_TEAM_ID']))
            away_abbr = ID_TO_ABBR.get(int(game['VISITOR_TEAM_ID']))

            if team_abbr in (home_abbr, away_abbr):
                is_home    = (team_abbr == home_abbr)
                opp_abbr   = away_abbr if is_home else home_abbr
                game_dt    = pd.Timestamp(check_date)
                days_rest  = int((game_dt - last_date).days)
                days_rest  = max(1, min(days_rest, 14))

                print(f"  Next game found: {check_date}  vs {opp_abbr}  "
                      f"({'Home' if is_home else 'Away'})  "
                      f"{days_rest} day(s) rest")
                return {
                    'date':         check_date.strftime('%Y-%m-%d'),
                    'opponent_abbr': opp_abbr,
                    'is_home':      is_home,
                    'days_rest':    days_rest,
                }

        time.sleep(0.35)

    print("  No game found in the next 14 days.")
    return None

# ─────────────────────────────────────────────────────────────────────────────
# Edit these two lines.  Everything else is auto-detected.
PLAYER_ID  = 1629029   # Luka Doncic  — change to any ID in PLAYERS
TARGET_PTS = 30        # P(score >= this many points)?  Set to None to skip.
# ─────────────────────────────────────────────────────────────────────────────

# Optional manual overrides — set to None to use the auto-detected value
OVERRIDE_OPPONENT  = None   # e.g. 'BOS' to model a specific hypothetical match-up
OVERRIDE_HOME      = None   # True/False
OVERRIDE_DAYS_REST = None   # integer

# Auto-fetch next game from schedule
print(f"Looking up next game for {PLAYERS.get(PLAYER_ID, PLAYER_ID)} ...")
schedule = fetch_next_scheduled_game(PLAYER_ID)

if schedule:
    opponent   = OVERRIDE_OPPONENT  if OVERRIDE_OPPONENT  is not None else schedule['opponent_abbr']
    is_home    = OVERRIDE_HOME      if OVERRIDE_HOME       is not None else schedule['is_home']
    days_rest  = OVERRIDE_DAYS_REST if OVERRIDE_DAYS_REST  is not None else schedule['days_rest']
else:
    # Fallback to sensible defaults if schedule lookup fails
    print("Schedule lookup failed — using defaults (away, 2 days rest, league-avg opponent).")
    opponent, is_home, days_rest = None, False, 2

result = predict_next_game(
    player_id      = PLAYER_ID,
    opponent_abbr  = opponent,
    is_home        = is_home,
    days_rest_next = days_rest,
    target_pts     = TARGET_PTS,
)


In [ ]:
# ── Bet value calculator ───────────────────────────────────────────────────────
#
# Odds formats supported:
#   Fractional : '20/21'  '5/4'  '10/11'   (UK / Betfair style)
#   American   : '+150'   '-110'            (DraftKings / FanDuel style)
#   Decimal    :  2.50     1.91             (Bet365 / European style)
#
# MODEL_PROB : auto-filled from result['prob_exceed'] — or set manually (0.0–1.0)
# ODDS       : paste exactly what the sportsbook shows  e.g. '20/21', '+135', '2.50'
# STAKE      : amount you are considering betting
# BANKROLL   : total bankroll (used for Kelly sizing)

MODEL_PROB = result['prob_exceed'] / 100 if result and result['prob_exceed'] else 0.40
ODDS       = '20/21'   # ← paste sportsbook odds here
STAKE      = 100       # £/$ amount
BANKROLL   = 1000      # £/$ total bankroll

# ─────────────────────────────────────────────────────────────────────────────

def parse_odds(odds_input):
    """
    Convert any common odds format to decimal odds.
      Fractional  '20/21' or '5/4'  → numerator/denominator + 1
      American    '+150' or '-110'   → standard conversion
      Decimal      2.50              → returned as-is
    """
    s = str(odds_input).strip().replace(' ', '')

    # Fractional: contains '/' and no leading sign
    if '/' in s:
        num, den = s.split('/')
        return float(num) / float(den) + 1

    # American: starts with + or is a negative integer (no decimal point)
    if s.startswith('+') or (s.startswith('-') and '.' not in s):
        american = int(s)
        return (american / 100 + 1) if american > 0 else (100 / abs(american) + 1)

    # Decimal
    return float(s)

def to_fractional(dec):
    """Convert decimal odds back to a tidy fractional string for display."""
    from fractions import Fraction
    frac = Fraction(dec - 1).limit_denominator(100)
    return f"{frac.numerator}/{frac.denominator}"

def to_american(dec):
    if dec >= 2.0:
        return f"+{int(round((dec - 1) * 100))}"
    else:
        return f"-{int(round(100 / (dec - 1)))}"

def bet_analysis(model_prob, odds_input, stake=100, bankroll=1000):
    p   = float(model_prob)
    q   = 1 - p
    dec = parse_odds(odds_input)
    b   = dec - 1                      # net profit per £1 staked

    implied_prob = 1 / dec
    edge         = p - implied_prob

    ev_per_unit  = p * b - q
    ev_stake     = ev_per_unit * stake

    kelly_f      = max(0, (p * b - q) / b)
    half_kelly   = kelly_f / 2
    kelly_stake  = half_kelly * bankroll

    verdict = (
        "BET  — positive edge"       if edge > 0.05  else
        "LEAN BET  — marginal edge"  if edge > 0.01  else
        "PASS  — no edge"            if edge >= -0.01 else
        "PASS  — book has the edge"
    )

    print(f"\n{'='*54}")
    print(f"  Bet Value Analysis")
    print(f"{'='*54}")
    print(f"  Sportsbook odds      : {to_fractional(dec):<8} "
          f"{to_american(dec):<8}  decimal {dec:.3f}")
    print(f"{'─'*54}")
    print(f"  Model probability    : {p*100:.1f}%")
    print(f"  Implied probability  : {implied_prob*100:.1f}%   (from odds)")
    print(f"  Edge                 : {edge*100:+.1f}%")
    print(f"{'─'*54}")
    print(f"  EV per £1 staked     : {ev_per_unit:+.3f}")
    print(f"  EV on £{stake} bet     : {ev_stake:+.2f}")
    print(f"{'─'*54}")
    print(f"  Kelly stake (half)   : £{kelly_stake:.0f}  "
          f"({half_kelly*100:.1f}% of £{bankroll} bankroll)")
    print(f"{'─'*54}")
    print(f"  Verdict              : {verdict}")
    print(f"{'='*54}")

    if edge <= 0:
        print("\n  The book's implied probability exceeds the model's estimate.")
        print("  No mathematical edge — pass on this bet.")
    elif kelly_stake < 5:
        print("\n  Edge exists but is thin. Kelly suggests a very small stake.")

    return {'edge': round(edge, 4), 'ev': round(ev_stake, 2),
            'kelly_stake': round(kelly_stake, 2), 'verdict': verdict}

bet_result = bet_analysis(MODEL_PROB, ODDS, STAKE, BANKROLL)
